**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# LLMs from the Ground Up

Every piece of a modern language model, built small enough to train on a laptop during the session: tokenization, embeddings, the next-token objective, and the inference tricks (temperature, top-k, KV caching) that turn a trained network into a chatbot's engine. The [transformer architecture itself](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) is a prerequisite — here we focus on the *language modeling* around it.

## 1. Pre-requisites

- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — attention.
- [Training Dynamics](./Training_Dynamics.ipynb) — Adam, schedules.
- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) — cross-entropy: an LLM is literally trained to *compress text*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
torch.manual_seed(0)

# our corpus: a tiny fable, repeated with variations — small enough to learn in minutes
base = (
"the fox watched the river. the river carried leaves and light. "
"a heron stood in the shallows and waited for fish. "
"the fox wanted fish too, but the fox could not wade. "
"so the fox watched the heron, and the heron watched the water. "
"when the fish rose, the heron struck. the fox learned patience from the heron. "
"in the morning the river was silver. in the evening the river was gold. "
"the leaves drifted, the light faded, and the fox went home with an idea. "
)
text = base * 40                      # ~11k characters
print(f"corpus: {len(text):,} characters")

---
### 🕐 Session 1 of 4 — *Tokens & Embeddings* (~35 min)
**Goal:** turn text into integers, integers into vectors; understand what BPE buys real models.
**Builds on:** [Transformers workshop](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (the pretraining objective).

---

## 2. Text → Numbers

💡 **Intuition.** A model eats vectors, not letters. Step 1 — **tokenize**: chop text into pieces from a fixed vocabulary and number them. We use characters (simple, small vocab); real LLMs use **BPE** — start from characters, repeatedly merge the most frequent adjacent pair ('t'+'h'→'th', 'th'+'e'→'the'), until common words are single tokens and rare words split into parts. It's a *compression* scheme ([Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb)!): frequent strings get short codes. Step 2 — **embed**: a learned lookup table maps each token id to a vector; during training, tokens used similarly drift together.

In [ ]:

# YOUR CODE HERE


In [ ]:
# mini-BPE, 12 merges, to see the mechanism real tokenizers scale up

# YOUR CODE HERE


---
### 🕐 Session 2 of 4 — *Pretraining: the Next-Token Objective* (~40 min)
**Goal:** train a small GPT on next-character prediction; watch loss approach the corpus entropy.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (finetuning at a glance).

---

## 3. One Objective To Rule Them All

💡 **Intuition.** The entire pretraining recipe is: *predict the next token, everywhere, forever*. The loss is cross-entropy — so training literally minimizes the bits needed to encode the corpus ([source coding](../Intro_Math/Information_Theory/Information_Theory.ipynb)): **an LLM is a learned compressor**, and everything it 'knows' exists because knowing it helps compression. Grammar helps predict; facts help predict; style helps predict. Scale the corpus and the model, and the compressor is forced to become a world-modeler.

In [ ]:

# YOUR CODE HERE


In [ ]:
# corpus entropy baselines: what loss SHOULD we expect?

# YOUR CODE HERE


In [ ]:

# YOUR CODE HERE


In [ ]:

# YOUR CODE HERE


---
### 🕐 Session 3 of 4 — *Finetuning & Alignment, at a Glance* (~30 min)
**Goal:** from raw predictor to assistant: SFT, preference learning, and what they change.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (inference).

---

## 4. From Predictor to Assistant

A pretrained model only continues text. Turning it into an assistant is *further training with different data*:

1. **Supervised finetuning (SFT)** — same next-token loss, but on curated (instruction → good response) pairs. The model learns the *format* of being helpful.
2. **Preference tuning (RLHF/DPO)** — humans rank pairs of responses; the model is pushed toward preferred ones. This shapes *judgment*, not knowledge: pretraining knows, alignment chooses.

💡 **Intuition.** Pretraining is the library; finetuning is the librarian's training. Both use gradient descent; only the data — and therefore what's being compressed — changes. We can demo the *mechanism* in miniature: finetune our fable model on a different style and watch the voice change.

In [ ]:
# 'SFT' in miniature: continue training on a new style — terse telegrams

# YOUR CODE HERE


---
### 🕐 Session 4 of 4 — *Inference: Sampling & the KV Cache* (~35 min)
**Goal:** temperature and top-k as knobs on a distribution; why caching makes generation O(1) per token.
**Builds on:** Sessions 2–3.

---

## 5. Serving the Model

💡 **Intuition.** **Sampling knobs.** The model outputs a *distribution*; how you draw from it sets the personality. Temperature $T$ rescales logits before softmax: $T \to 0$ is argmax (deterministic, repetitive), $T > 1$ flattens (creative, error-prone). Top-k truncates to the $k$ most likely before sampling — a guardrail against the long tail of nonsense.

**The KV cache.** Naive generation re-runs the whole prefix for every new token — $O(n^2)$ pain. But causal attention means old tokens' keys/values *never change*: cache them, and each new token costs one attention row. This single trick is why chatbots stream tokens at constant speed — and why long contexts eat GPU memory (the cache IS the memory hog).

In [ ]:

# YOUR CODE HERE


In [ ]:
# measure the quadratic blowup the KV cache exists to kill (our model recomputes the prefix)

# YOUR CODE HERE


## 6. Conclusion

Tokenize (compressively), embed, predict-the-next-token until the loss approaches the corpus's entropy, finetune to choose a voice, then sample with temperature/top-k behind a KV cache. Everything else about LLMs is *scale* — which you studied in [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb).

---
## Where next

- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — the architecture inside `self.blocks`.
- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) — the compression view, formalized.
- [Model Compression](./Model_Compression.ipynb) — fitting these onto real hardware.